# HC + RoBERTa Concat — 최종 실험
**담당: 김광빈**

데이터: `data_yelp_features.parquet` (최종본, 전처리 완료)  
HC: 23개 (LIWC 5개 제거: anger, sadness, posemo, anx, negate)  
스케일링: log(x+1)  
임베딩: roberta-base 직접 추출

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import random
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


## 1. 데이터 로드

In [2]:
df = pd.read_parquet('/content/data_yelp_features.parquet')
print('shape:', df.shape)
print('label 분포:')
print(df['label'].value_counts())

# label 변환
df['label_int'] = (df['label'] == 'ai').astype(int)

# HC 피쳐 (LIWC 5개 제거)
REMOVE_COLS = ['anger', 'sadness', 'posemo', 'anx', 'negate']
NON_HC = ['pk', 'review_id', 'text', 'label', 'label_int', 'source', 'review_stars', 'business_id']
HC_COLS = [c for c in df.columns if c not in NON_HC and c not in REMOVE_COLS]
HC_DIM = len(HC_COLS)
print(f'\nHC 피쳐 {HC_DIM}개:', HC_COLS)

shape: (20000, 35)
label 분포:
label
human    10000
ai       10000
Name: count, dtype: int64

HC 피쳐 23개: ['syllable', 'lexicon', 'sentence', 'char', 'letter', 'polysyllab', 'monosyllab', 'smog_index', 'flesch_reading_ease', 'flesch_kincaid_grade', 'fog_scale', 'dale_chall', 'reading_time', 'sentiment', 'subjectivity', 'perplexity', 'burstiness', 'nouns', 'adj', 'verbs', 'pronoun', 'adverb', 'article']


## 2. RoBERTa 임베딩 추출 (~30-40분)

In [3]:
ROBERTA_MODEL = 'roberta-base'
rob_tok = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
rob_model = AutoModel.from_pretrained(ROBERTA_MODEL).to(DEVICE).eval()

@torch.no_grad()
def extract_embeddings(texts, batch_size=64, max_length=256):
    all_emb = []
    for i in tqdm(range(0, len(texts), batch_size), desc='embedding'):
        batch = [str(t) for t in texts[i:i+batch_size]]
        enc = rob_tok(batch, return_tensors='pt', truncation=True,
                      padding=True, max_length=max_length).to(DEVICE)
        out = rob_model(**enc)
        all_emb.append(out.last_hidden_state[:, 0].cpu().numpy())  # CLS
    return np.vstack(all_emb)

embeddings = extract_embeddings(df['text'].tolist())
print('embeddings shape:', embeddings.shape)
np.save('roberta_embeddings_final.npy', embeddings)
print('저장 완료: roberta_embeddings_final.npy')

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


embedding:   0%|          | 0/313 [00:00<?, ?it/s]

embeddings shape: (20000, 768)
저장 완료: roberta_embeddings_final.npy


## 3. HC 스케일링 (log(x+1)) + Train/Val/Test Split

In [4]:
# HC log(x+1) 스케일링
hc_raw = df[HC_COLS].values.astype(np.float32)
hc_scaled = np.log1p(np.abs(hc_raw)) * np.sign(hc_raw)  # log(x+1) 적용

labels = df['label_int'].values

# 70:15:15 split
set_seed()
idx = np.arange(len(labels))
idx_train, idx_temp = train_test_split(idx, test_size=0.3, stratify=labels, random_state=SEED)
idx_val,   idx_test = train_test_split(idx_temp, test_size=0.5, stratify=labels[idx_temp], random_state=SEED)
print(f'Train {len(idx_train)} | Val {len(idx_val)} | Test {len(idx_test)}')

emb_train, emb_val, emb_test = embeddings[idx_train], embeddings[idx_val], embeddings[idx_test]
hc_train,  hc_val,  hc_test  = hc_scaled[idx_train], hc_scaled[idx_val], hc_scaled[idx_test]
y_train,   y_val,   y_test   = labels[idx_train], labels[idx_val], labels[idx_test]

Train 14000 | Val 3000 | Test 3000


## 4. DataLoader

In [5]:
def make_loaders(batch_size):
    def to_ds(emb, hc, y):
        return TensorDataset(
            torch.tensor(emb, dtype=torch.float32),
            torch.tensor(hc,  dtype=torch.float32),
            torch.tensor(y,   dtype=torch.long)
        )
    return (
        DataLoader(to_ds(emb_train, hc_train, y_train), batch_size=batch_size, shuffle=True),
        DataLoader(to_ds(emb_val,   hc_val,   y_val),   batch_size=batch_size),
        DataLoader(to_ds(emb_test,  hc_test,  y_test),  batch_size=batch_size),
    )
print('완료')

완료


## 5. 모델 정의

In [6]:
class HCRoBertaConcatModel(nn.Module):
    """RoBERTa CLS [768] + HC [23] → concat → MLP → 2"""
    def __init__(self, emb_dim=768, hc_dim=HC_DIM, dmodel=256, dropout=0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + hc_dim, dmodel), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dmodel, 64),               nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )
    def forward(self, emb, hc):
        return self.mlp(torch.cat([emb, hc], dim=1))

print('모델 정의 완료')

모델 정의 완료


## 6. 학습 / 평가 함수

In [7]:
LR_GRID      = [3e-5, 1e-4, 3e-4]
DROPOUT_GRID = [0.0, 0.1, 0.3]
DMODEL_GRID  = [128, 256, 512]
BATCH_GRID   = [32, 64, 128]
MAX_EPOCHS   = 30
PATIENCE     = 5
DEFAULT      = dict(lr=1e-4, dropout=0.1, dmodel=256, batch=64)

def train_epoch(model, loader, optimizer, criterion):
    model.train(); total = 0
    for emb, hc, y in loader:
        emb, hc, y = emb.to(DEVICE), hc.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(emb, hc), y)
        loss.backward(); optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def eval_loop(model, loader, criterion):
    model.eval()
    preds, probs, ys, losses = [], [], [], []
    for emb, hc, y in loader:
        emb, hc, y = emb.to(DEVICE), hc.to(DEVICE), y.to(DEVICE)
        logits = model(emb, hc)
        losses.append(criterion(logits, y).item())
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
        preds.extend(logits.argmax(1).cpu().numpy())
        ys.extend(y.cpu().numpy())
    return {
        'val_loss':  round(float(np.mean(losses)), 4),
        'accuracy':  round(accuracy_score(ys, preds), 4),
        'precision': round(precision_score(ys, preds), 4),
        'recall':    round(recall_score(ys, preds), 4),
        'f1':        round(f1_score(ys, preds), 4),
        'auroc':     round(roc_auc_score(ys, probs), 4),
    }

def train_model(model, hparams, train_loader, val_loader, test_loader, verbose=True):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=hparams['lr'])
    criterion = nn.CrossEntropyLoss()
    best_val_loss, best_state, patience_cnt, best_epoch = float('inf'), None, 0, 1

    for ep in range(1, MAX_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion)
        vm = eval_loop(model, val_loader, criterion)
        if verbose:
            print(f'  Ep{ep:2d} | tr={tr_loss:.4f} | val_loss={vm["val_loss"]:.4f} | F1={vm["f1"]:.4f}')
        if vm['val_loss'] < best_val_loss:
            best_val_loss = vm['val_loss']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0; best_epoch = ep
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                if verbose: print(f'  Early stop @ ep{ep} (best={best_epoch})')
                break

    model.load_state_dict(best_state)
    tm = eval_loop(model, test_loader, criterion)
    tm['best_epoch'] = best_epoch
    return tm

print('함수 준비 완료')

함수 준비 완료


## 10. 비교 모델: HC+RoBERTa Cross-Attention

HC [23] → 길이-1 시퀀스로 취급 → RoBERTa CLS와 Cross-Attention  
(Concat과의 성능 비교용 베이스라인)

In [11]:
class HCRoBertaCrossAttnModel(nn.Module):
    """
    Q  = RoBERTa CLS [B, 1, dmodel]  (텍스트가 HC를 참조)
    K  = V = HC projected [B, 1, dmodel]
    → cross-attn → residual + LN → MLP → 2
    """
    def __init__(self, emb_dim=768, hc_dim=HC_DIM, dmodel=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.emb_proj   = nn.Linear(emb_dim, dmodel)
        self.hc_proj    = nn.Linear(hc_dim,  dmodel)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=dmodel, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(dmodel)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(dmodel, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, emb, hc):
        q  = self.emb_proj(emb).unsqueeze(1)   # [B, 1, dmodel]
        kv = self.hc_proj(hc).unsqueeze(1)     # [B, 1, dmodel]
        attn_out, _ = self.cross_attn(q, kv, kv)
        out = self.norm(q + self.drop(attn_out)).squeeze(1)  # [B, dmodel]
        return self.classifier(out)

print('CrossAttn 모델 정의 완료')

CrossAttn 모델 정의 완료


In [12]:
NUM_HEADS_GRID = [2, 4, 8]
CA_DEFAULT     = dict(lr=1e-4, dropout=0.1, dmodel=256, batch=64, num_heads=4)

def greedy_search_ca():
    best = CA_DEFAULT.copy()
    print('\n======== Greedy Search: HC+RoBERTa Cross-Attention ========\n')
    for param, grid in [
        ('lr',        LR_GRID),
        ('dropout',   DROPOUT_GRID),
        ('dmodel',    DMODEL_GRID),
        ('batch',     BATCH_GRID),
        ('num_heads', NUM_HEADS_GRID),
    ]:
        print(f'--- {param} ---')
        best_f1 = -1
        for val in grid:
            hp = best.copy(); hp[param] = val
            set_seed()
            tr, vl, te = make_loaders(hp['batch'])
            m = HCRoBertaCrossAttnModel(dmodel=hp['dmodel'], num_heads=hp['num_heads'], dropout=hp['dropout'])
            tm = train_model(m, hp, tr, vl, te, verbose=False)
            print(f'  {param}={val}: F1={tm["f1"]:.4f} AUROC={tm["auroc"]:.4f}')
            if tm['f1'] > best_f1: best_f1 = tm['f1']; best[param] = val
        print(f'  → best {param}={best[param]}\n')
    print('최적 HP:', best)
    return best

best_hp_ca = greedy_search_ca()


======== Greedy Search: HC+RoBERTa Cross-Attention ========

--- lr ---
  lr=3e-05: F1=0.9815 AUROC=0.9989
  lr=0.0001: F1=0.9831 AUROC=0.9989
  lr=0.0003: F1=0.9821 AUROC=0.9988
  → best lr=0.0001

--- dropout ---
  dropout=0.0: F1=0.9826 AUROC=0.9988
  dropout=0.1: F1=0.9831 AUROC=0.9989
  dropout=0.3: F1=0.9824 AUROC=0.9988
  → best dropout=0.1

--- dmodel ---
  dmodel=128: F1=0.9834 AUROC=0.9989
  dmodel=256: F1=0.9831 AUROC=0.9989
  dmodel=512: F1=0.9799 AUROC=0.9982
  → best dmodel=128

--- batch ---
  batch=32: F1=0.9827 AUROC=0.9989
  batch=64: F1=0.9834 AUROC=0.9989
  batch=128: F1=0.9824 AUROC=0.9988
  → best batch=64

--- num_heads ---
  num_heads=2: F1=0.9834 AUROC=0.9989
  num_heads=4: F1=0.9834 AUROC=0.9989
  num_heads=8: F1=0.9837 AUROC=0.9989
  → best num_heads=8

최적 HP: {'lr': 0.0001, 'dropout': 0.1, 'dmodel': 128, 'batch': 64, 'num_heads': 8}


In [13]:
print(f'\n======== 5-Run: HC+RoBERTa Cross-Attention | {best_hp_ca} ========\n')
tr, vl, te = make_loaders(best_hp_ca['batch'])
rows_ca = []
for seed in range(42, 47):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    m = HCRoBertaCrossAttnModel(dmodel=best_hp_ca['dmodel'], num_heads=best_hp_ca['num_heads'], dropout=best_hp_ca['dropout'])
    tm = train_model(m, best_hp_ca, tr, vl, te, verbose=False)
    print(f'  Seed {seed}: Acc={tm["accuracy"]} | Prec={tm["precision"]} | Recall={tm["recall"]} | F1={tm["f1"]} | AUROC={tm["auroc"]} | best_epoch={tm["best_epoch"]}')
    rows_ca.append({'Seed': seed, **tm})

runs_ca_df = pd.DataFrame(rows_ca)
print(f'\n평균 F1={runs_ca_df["f1"].mean():.4f} ± {runs_ca_df["f1"].std():.4f}')
print(f'평균 AUROC={runs_ca_df["auroc"].mean():.4f} ± {runs_ca_df["auroc"].std():.4f}')
print(f'\n비고:')
print(f'best_epoch={int(runs_ca_df["best_epoch"].mean())}')
print(f'learning_rate={best_hp_ca["lr"]}')
print(f'dropout={best_hp_ca["dropout"]}')
print(f'batch_size={best_hp_ca["batch"]}')
print(f'dmodel={best_hp_ca["dmodel"]}')
print(f'num_heads={best_hp_ca["num_heads"]}')


======== 5-Run: HC+RoBERTa Cross-Attention | {'lr': 0.0001, 'dropout': 0.1, 'dmodel': 128, 'batch': 64, 'num_heads': 8} ========

  Seed 42: Acc=0.9837 | Prec=0.9808 | Recall=0.9867 | F1=0.9837 | AUROC=0.9989 | best_epoch=22
  Seed 43: Acc=0.982 | Prec=0.9769 | Recall=0.9873 | F1=0.9821 | AUROC=0.9989 | best_epoch=16
  Seed 44: Acc=0.9837 | Prec=0.9866 | Recall=0.9807 | F1=0.9836 | AUROC=0.9989 | best_epoch=27
  Seed 45: Acc=0.9837 | Prec=0.9821 | Recall=0.9853 | F1=0.9837 | AUROC=0.9989 | best_epoch=30
  Seed 46: Acc=0.983 | Prec=0.9776 | Recall=0.9887 | F1=0.9831 | AUROC=0.9989 | best_epoch=23

평균 F1=0.9832 ± 0.0007
평균 AUROC=0.9989 ± 0.0000

비고:
best_epoch=23
learning_rate=0.0001
dropout=0.1
batch_size=64
dmodel=128
num_heads=8
